# 300391 / 20260320：V0 复牌后的限价单可见性诊断
输入为原始 Parquet 的单证券提取，不改状态、时间或价量。伴随脚本逐原生序号检查委托生命周期，并独立模拟当前全部为限价单的 HideIfCrossing 路径。该模拟仅用于定位差异，不是新的交易所规则实现。
首个候选序号 27304188：独立余量账本的两侧十档价量、委托数和总量与原始 T0 完全一致；当前隐藏模型复现 Rust 缺失的 0.46 元 / 14 笔 / 643600 股。14 笔在该候选之前均无成交或撤单。
官方依据：[波动临停期间可申报撤单，复牌时实行盘中集合竞价](https://investor.szse.cn/index/update/t20200826_581025.html)。该规则不保证不同 feed 的时间戳相同。
本次未改生产代码。当前 release 二进制对原值提取 fixture 复验为 3688 匹配、102 不匹配（101 T0、1 E0）；这不是全市场验收。E0 是另一个有效竞价范围问题。

In [ ]:
from pathlib import Path
import runpy
root = Path.cwd()
if root.name == 'analysis':
    root = root.parent
audit = runpy.run_path(str(root / 'analysis/audit_sz_v0_reentry.py'))
result = audit['run']()
assert result['checks']['independent_ledger_depth_counts_totals_match_reference']
[(o['ApplSeqNum'], o['OrderQty'], o['first_later_execution']) for o in result['candidate']['hidden_orders']]